# 05 — Train Conv-LoRA on MedSAM2

Same injection points as LoRA, but the down-projection is followed by a
lightweight depth-wise 3×3 conv over the spatial token grid before the
up-projection. This adds a local image prior, which the Conv-LoRA paper
argues helps on fine-structured / domain-shifted segmentation — a good
match for carbon-fibre filaments.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from cfrp_medsam2.train import TrainConfig, train
from cfrp_medsam2.model import ModelConfig
from cfrp_medsam2.lora import LoRAConfig

REPO = Path('..').resolve()

cfg = TrainConfig(
    regime='conv_lora',
    model=ModelConfig(
        backend='medsam2',
        checkpoint=str(REPO / 'checkpoints' / 'sam2.1_hiera_tiny.pt'),
        image_size=512,
    ),
    lora=LoRAConfig(
        rank=8,
        alpha=16.0,
        dropout=0.05,
        target_substrings=('qkv', 'q_proj', 'k_proj', 'v_proj', 'out_proj', 'proj'),
        exclude_substrings=('mask_decoder.iou_prediction_head', 'mlp', 'obj_ptr'),
        use_conv=True,
        conv_kernel=3,
        train_mask_decoder=True,
        include_memory_attention=True,
    ),
    train_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('train_*.npz'))),
    val_volumes=tuple(str(p) for p in sorted((REPO / 'data' / 'processed' / 'toy').glob('val_*.npz'))),
    image_size=512,
    batch_size=1,
    epochs=5,
    lr=1.0e-4,
    prompt_mode='mixed',
    ckpt_dir=str(REPO / 'checkpoints'),
    log_path=str(REPO / 'logs' / 'conv_lora.csv'),
)
result = train(cfg)
print('best_val_dice =', result['best_val_dice'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(REPO / 'logs' / 'conv_lora.csv')
df.plot(x='epoch', y=['train_dice', 'val_dice'], figsize=(6, 3.5))
plt.title('Conv-LoRA training curves')
plt.grid(True, alpha=0.3)